# Appendix: TRF Repeat Panel Construction

This appendix documents how the repeat locus panel used in Notebook 7 was built.
[Tandem Repeat Finder](https://tandem.bu.edu/trf/trf.html) (TRF, Benson 1999) is a
published third-party tool — this appendix covers installation, the command-line
invocation used in this paper, and the filtering steps that yield the curated panel.

Parsing and filtering functions are implemented in `nwflex.trf`.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from nwflex.trf import (
    parse_trf_dat,
    annotate_repeat_isolation,
    filter_isolated_repeats
)

DATA_DIR = Path("../data")
PANEL_TSV = DATA_DIR / "wgs.motif_catalog_K100.panel.v2.tsv"
print("Panel exists:", PANEL_TSV.exists())

Panel exists: True


## Running TRF

We use [Tandem Repeat Finder](https://tandem.bu.edu/trf/trf.html) (TRF) to identify
repetitive regions in the reference genome. TRF scans a FASTA for tandem repeats and
produces a `.dat` file with one row per detected repeat tract: coordinates, period
size, copy number, percent matches, and the consensus motif sequence.

### Installation

```bash
conda install -c bioconda trf
```

**samtools** is also needed to index the reference FASTA
(only needed if running from your own genome; skip if using the pre-built panel):

```bash
conda install -c bioconda samtools
```

### Running TRF

The command below runs TRF on a single chromosome. The numeric parameters control
the match/mismatch weights and the minimum alignment score — these are the standard
settings used throughout this paper:

```bash
trf hg38.chr21.fa 2 5 5 80 10 20 100 -f -d -m -ngs > chr21.trf.dat
#              ^ match weight
#                ^ mismatch penalty
#                  ^ indel penalty
#                     ^^ match probability (for scoring)
#                        ^^ indel probability
#                           ^^ minimum alignment score
#                              ^^^ maximum period size (bp)
```

Run this for each chromosome of interest. Outputs land in
`genomes/trf/chroms/chr<N>.trf.dat`.

## Parsing and filtering

The raw TRF output includes nested, imperfect, and closely-spaced repeats. For our
simulations, we keep only **isolated, perfect** loci — those with at least 100 bp of
uninterrupted flank on each side and no overlapping neighbors.

Three functions from `nwflex.trf` handle this:

- `parse_trf_dat(path)` — reads a `.dat` file into a DataFrame, one row per detected repeat
- `annotate_repeat_isolation(df, chrom_lengths)` — adds `ldist` / `rdist` (distance to nearest neighbor) and per-locus overlap count
- `filter_isolated_repeats(df, ...)` — keeps only loci that pass isolation, purity, and period-size criteria

In [2]:
# Demo: parse the included fixture (a chr21 snippet)
_fixture = DATA_DIR / "chr21_snippet.dat"
_raw = parse_trf_dat(_fixture)
print(f"Raw TRF rows: {len(_raw)}")

_annotated = annotate_repeat_isolation(_raw, {"chr21": 46_709_983})
_filtered = filter_isolated_repeats(_annotated, min_dist=100)
print(f"After isolation filter (min_dist=100 bp): {len(_filtered)} loci")
# The real panel further restricts to max_period=2 (mono/dinucleotide);
# this small fixture has no such loci that are also isolated.

_filtered[["chrom", "start", "end", "period_size", "copy_number",
           "consensus_pattern", "ldist", "rdist"]]

Raw TRF rows: 10
After isolation filter (min_dist=100 bp): 4 loci


,chrom,start,end,period_size,copy_number,consensus_pattern,ldist,rdist
0,chr21,5016247,5016270,3,7.7,GAG,5016247,305
1,chr21,5016575,5016590,5,3.0,GTCCT,305,111
6,chr21,5017442,5017453,5,2.2,GCAGG,513,159
8,chr21,5017612,5017782,83,2.0,CAAAAGTACAGGACCTCAGCCTTGGCAGACAAAGGAGGGACCTGCT...,159,41692201


## The pre-built panel

For each mono-, di-, and trinucleotide repeat, we selected up to 100 loci from the human
genome hg38. The curated panel is at `data/wgs.motif_catalog_K100.panel.v2.tsv` in the
repository.

Loci are perfect and isolated (≥ 100 bp uninterrupted flank on each side, no
overlapping neighbors). The `lflank` and `rflank` columns hold the raw genomic
sequence flanking the repeat; Notebook 7 takes the innermost `FLANK_LEN` bp from
each side for simulation.

The K100 panel was built in two stages:

1. Build a large iso-pure v2 panel from TRF calls on the chromosomes with available `.dat` files:

```bash
PYTHONPATH=stormflex/src python stormflex/wgs/scripts/make_panel_fast.py \
    --chr chr1 chr2 chr4 chr6 chr8 chr10 chr12 chr13 chr14 chr15 chr16 chr19 chr20 chr21 chr22 chrY \
    --fasta genomes/hg38.chrYpsr.fa \
    --trf-dir genomes/trf/chroms \
    --require-pure \
    --max-period 6 \
    --max-tract-bp 1000 \
    --output /data/safe/levy/genome/wgs-panels/wgs.16chr.iso_pure.panel.v2.tsv
```

2. Take the first 100 boundary-clean loci per mono-, di-, and trinucleotide motif:

```bash
python harness/build_motif_catalog.py \
    --panel-tsv /data/safe/levy/genome/wgs-panels/wgs.16chr.iso_pure.panel.v2.tsv \
    --K 100 \
    --max-period 3 \
    --output data/wgs.motif_catalog_K100.panel.v2.tsv
```

Some CG-containing motifs have fewer than 100 loci after the filters, so the
final catalog contains 6,900 loci rather than exactly 7,600.

In [3]:
if not PANEL_TSV.exists():
    raise FileNotFoundError(
        f"Panel not found at {PANEL_TSV}\n"
        "Run this notebook from the notebooks/ directory, or update DATA_DIR in the setup cell."
    )

panel = pd.read_csv(PANEL_TSV, sep="\t")
print(f"{len(panel)} loci in pre-built panel")
print("Columns:", list(panel.columns))
panel.head(3)

6900 loci in pre-built panel
Columns: ['pind', 'chr', 'start_38', 'stop_38', 'strand', 'type', 'lflank', 'rflank', 'ms_seq', 'ref_score_per_base']


,pind,chr,start_38,stop_38,strand,type,lflank,rflank,ms_seq,ref_score_per_base
0,0,chr1,36351,36364,+,A,CCATCTCTGGGCCCAGAATGACCCACTGGAGACCTTACAGCTCTCC...,CCCAGCCTGGCGGAAAGAATTTAAATTATAAAAACTTAGAAGTATG...,AAAAAAAAAAAAA,1.0
1,1,chr1,51864,51877,+,A,GTGGGGGTTGAGTTTCACTTTATTTAAAGTGAGTCTTAATCCTCCA...,GAAGATTGATCAGAGAGTACCTCCCCTAAGGGTACATGCAGATAAA...,AAAAAAAAAAAAA,1.0
2,2,chr1,71175,71186,+,A,AGTATATTACTTGGATCCATCTATGTCATTTTCCATGGTTAATGTT...,CCTTAACAAATGATTCTGACAAATATCTTCTCTTTCCAGGGAGAAT...,AAAAAAAAAAA,1.0
